#Лабораторная работа 7. Обучение модели T5

**Задание 1**. Изучите и запустите код ниже. Ответьте на вопросы в конце.

##Добучение модели `google/long-t5-tglobal-base` на датасете `SQUAD`.

###Некоторые комментарии:
- будем использовать только небольшую часть датасета `SQUAD`, чтобы уложиться во временные рамки занятий,
- считается, что модели T5 (кроме T5-small) имеют проблемы с mixed precision training. При использовании `bf16=True` или `fp16=True` возникают `nan` в градиентах, поэтому не используйте смешанныую точность при обучении,
- обратите внимание на маскирование labels значением `-100`,
- используйте `report_to="none"` в аргументах обучения, чтобы не логировать процесс обучения в системе `wandb`,
- сохраняйте обученную модель, чтобы не запускать заново обучение

In [1]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


Делаем импорт библиотек

In [2]:
import torch
from transformers import (
    AutoTokenizer,
    LongT5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import load_dataset, DatasetDict
import numpy as np
from evaluate import load

Загружаем модель и токенизатор

In [3]:
model_checkpoint = "google/long-t5-tglobal-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = LongT5ForConditionalGeneration.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/297 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Датасет `SQUAD` относится к вопросно-ответному бенчмарку и содержит поля:
- вопрос,
- контекст, в котором содержится ответ,
- ответ.

In [4]:
def answer_question(question, context):
    """
    Генерация ответа на вопрос по контексту
    """
    input_text = f"question: {question} context: {context}" # text to text
    input_ids = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).input_ids
    #вернуть данные в формте тензора пайторч, возможность обрезки текста если больше 512
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    input_ids = input_ids.to(device)
    #перенос на видеокарту
    outputs = model.generate(
        input_ids,
        max_length=128,
        num_beams=4,#поиск по 4 лучам (4 вероятных ответа с выбором лучшего)
        early_stopping=True
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)#возвращаем разчифрованный тензор с числами без тех. токенов
    return answer

Модель `google/long-t5-tglobal-base` не предобучалась на датасете `SQUAD`. Посмотрим как она отвечает до дообучения.

In [5]:
test_context = """
The Amazon rainforest is a moist broadleaf tropical rainforest in the Amazon biome
that covers most of the Amazon basin of South America. The forest covers 5.5 million
square kilometers across nine countries.
"""
test_question = "How large is the Amazon rainforest?"

predicted_answer = answer_question(test_question, test_context)
print(f"\nТестовый пример:")
print(f"Вопрос: {test_question}")
print(f"Ответ модели: {predicted_answer}")


Тестовый пример:
Вопрос: How large is the Amazon rainforest?
Ответ модели: context: The Amazon rainforest is a moist broadleaf tropical rainforest in the Amazon biome that covers most of the Amazon basin of South America.


In [ ]:
'''
Модель не понимает, что нужно выделить краткий но информативный фрагмент текста.
Она продолжает или пересказывает текст, для того чтобы выполнить первое предложение нужно обучить модель
на датасете squad.
'''

In [6]:
dataset = load_dataset("squad")
print(f"Размер обучающей выборки: {len(dataset['train'])}")
print(f"Размер валидационной выборки: {len(dataset['validation'])}")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Размер обучающей выборки: 87599
Размер валидационной выборки: 10570


Создание подвыборки для обучения и валидации заданного размера.

In [7]:
train_subset = dataset["train"].shuffle(seed=42).select(range(2000))
val_subset = dataset["validation"].shuffle(seed=42).select(range(200))
dataset = DatasetDict({
    "train": train_subset,
    "validation": val_subset
})
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 200
    })
})

In [8]:
def preprocess_squad_for_t5(examples):
    """
    Преобразование SQuAD в формат text-to-text для T5:
    Input: "question: <вопрос> context: <контекст>"
    Target: "<ответ>"
    """
    inputs = []
    targets = []

    for question, context, answers in zip(
        examples["question"],
        examples["context"],
        examples["answers"]
    ):
    #формируем входные пары (вопрос и контекст, дай ответ)
        input_text = f"question: {question} context: {context}"
        inputs.append(input_text)

        target_text = answers["text"][0] if len(answers["text"]) > 0 else ""
        targets.append(target_text)

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )#то, что модель читает

    labels = tokenizer(
        targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )#то, что необходимо выдать в качестве правильного ответа

    labels["input_ids"] = [
        [(label if label != tokenizer.pad_token_id else -100) for label in labels_example]
        for labels_example in labels["input_ids"]
    ]#-100 - специальный сигнал для функции потерь: игнорируй токены и не считай их при расчете ошибки
    #чтобы модель не заполнялась пустышками
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [9]:
tokenized_dataset =  dataset.map(#запускаем на весь датасет сразу
    preprocess_squad_for_t5,
    batched=True,#берём по батчам а не по примерам
    remove_columns=dataset["train"].column_names #удаляем старые поля, теперь есть чистые токены labels и innput_ids
)

print(f"Пример токенизированных данных:")
print(tokenized_dataset["train"][0])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Пример токенизированных данных:
{'input_ids': [822, 10, 363, 5294, 13, 16341, 7, 5492, 15, 26, 380, 1687, 10736, 21, 273, 3140, 10172, 58, 2625, 10, 37, 1276, 210, 5841, 30, 18182, 3, 184, 2575, 2330, 13799, 10438, 38, 8, 8486, 6025, 684, 16, 8, 296, 21, 4761, 4333, 5, 37, 907, 1323, 3527, 30, 1331, 28789, 14179, 6, 3, 9, 2647, 18237, 2547, 3193, 13, 8, 837, 789, 6, 65, 2681, 10438, 30, 165, 1605, 570, 13, 1440, 24, 1457, 885, 4891, 788, 12, 8, 1405, 11, 5996, 13, 17880, 13, 4761, 4333, 5908, 16, 42, 21533, 26, 57, 8, 789, 5, 2150, 12, 3, 9, 2735, 1276, 210, 3699, 486, 6592, 7, 3719, 6, 505, 5988, 13, 16341, 7, 5492, 15, 26, 3510, 8, 1687, 10736, 21, 273, 113, 1175, 10172, 117, 489, 6170, 3510, 20070, 2462, 7, 11, 3753, 326, 13, 1780, 21, 14806, 11, 3, 5840, 1152, 63, 117, 11, 505, 5406, 380, 3, 4411, 53, 3, 9, 568, 113, 10042, 7, 3165, 4203, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [10]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model) #умный менеджер батчей, допишет padding нули до нужной длины

In [11]:
squad_metric = load("squad")

def compute_metrics(eval_pred):
    """
    Вычисление метрик Exact Match и F1
    """
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.argmax(predictions, axis=-1)
    #модель выдает токены считаем оценку по обучному тексту, переводим предсказания в прав. ответы обратно в текст
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)#возвращем служубные токены
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    formatted_predictions = [
        {"id": str(i), "prediction_text": pred}
        for i, pred in enumerate(decoded_preds)
    ]
    formatted_references = [
        {"id": str(i), "answers": {"text": [label], "answer_start": [0]}}
        for i, label in enumerate(decoded_labels)
    ]

    results = squad_metric.compute(#получит 1 если ответ совпал "буква в букву" - EM
        predictions=formatted_predictions,
        references=formatted_references
    )

    return {
        "exact_match": results["exact_match"],
        "f1": results["f1"]
    }

In [12]:
training_args = TrainingArguments(
    output_dir="./t5-squad-results", #куда сохранять результаты
    logging_strategy="epoch", #отчет и проверка один раз за эпоху
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=5e-5, #скорость обучения
    per_device_train_batch_size=8, #сколько примеров видит одновременно за один проход по gpu
    per_device_eval_batch_size=8,
    num_train_epochs = 3, #сколько раз модель целиком прочитает весь обучающий датасет
    fp16=False, #учится в полном формате fp32
    report_to="none",
    push_to_hub=False
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics = compute_metrics
)

In [14]:
print("Начало обучения...")
trainer.train()

Начало обучения...


Epoch,Training Loss,Validation Loss,Exact Match,F1
1,2.023380,1.052699,2.000000,7.743125
2,1.163330,0.829258,2.500000,6.966182
3,0.994706,0.795113,3.000000,7.175328


TrainOutput(global_step=750, training_loss=1.3938052368164062, metrics={'train_runtime': 1101.6742, 'train_samples_per_second': 5.446, 'train_steps_per_second': 0.681, 'total_flos': 4108713984000000.0, 'train_loss': 1.3938052368164062, 'epoch': 3.0})

In [15]:
print("\nОценка на валидационном наборе:")
results = trainer.evaluate()
print(f"Exact Match: {results['eval_exact_match']:.2f}")
print(f"F1 Score: {results['eval_f1']:.2f}")


Оценка на валидационном наборе:


Exact Match: 3.00
F1 Score: 7.18


In [16]:
model.save_pretrained("./t5-squad-finetuned")
tokenizer.save_pretrained("./t5-squad-finetuned")
print("\nМодель сохранена в ./t5-squad-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Модель сохранена в ./t5-squad-finetuned


In [17]:
test_context = """
The Amazon rainforest is a moist broadleaf tropical rainforest in the Amazon biome
that covers most of the Amazon basin of South America. The forest covers 5.5 million
square kilometers across nine countries.
"""
test_question = "How large is the Amazon rainforest?"

predicted_answer = answer_question(test_question, test_context)
print(f"\nТестовый пример:")
print(f"Вопрос: {test_question}")
print(f"Ответ модели: {predicted_answer}")


Тестовый пример:
Вопрос: How large is the Amazon rainforest?
Ответ модели: broadleaf tropical rainforest


**Ответьте на следующие вопросы**:
- Примеры какой задачи содержатся в датасете `SQUAD`?
- Чем полезен в какой момент обучения используется экземпляр класса `DataCollatorForSeq2Seq`?
- Сколько шагов выполняется в рамках обуечния одной эпохи? Как вычисляется количество шагов?
- Как задается размер батча?
- Какая метрика оценивания используется в приведённом примере дообучения на вопросно-ответной задаче?

In [ ]:
'''
1. Задача QA: на вход вопрос и контекст, в ответ получаем корректный ответ
2. Динамически добавляет нулевые токены к батчу (padding), чтобы все примеры имели одинаковую длиную,
правильно обрабатывает labels для моделей типа seq2seq
3. размер датасета делим на размер батча, то есть 2000 на 8, то получаем 250 шагов за эпоху
4. задаем через per_device_eval_batch_size=8,
5. EM - полное совпадение слов, F1 - частичное совпадение

'''

##Сравнение

**Задание 2**. Создайте новую валидационную выборку из датасета `SQUAD` и сравните на ней следующие три модели по метрикам, используемым выше:
- исходная модель `google/long-t5-tglobal-base`,
- сохранённая дообученная модель `google/long-t5-tglobal-base`,
- исходная модель `google-t5/t5-base`, которая предобучалась на датасете `SQUAD`.

Результаты оформите в виде таблицы.\
Сделайте выводы.

In [38]:
full_dataset = load_dataset("squad")
# 100 примеров из валидационного набора
test_dataset = full_dataset["validation"].shuffle(seed=42).select(range(100))

def prepare_test_data(examples):
    inputs = []
    targets = []
    for question, context, answers in zip(examples["question"], examples["context"], examples["answers"]):
        input_text = f"question: {question} context: {context}"
        inputs.append(input_text)
        target_text = answers["text"][0] if len(answers["text"]) > 0 else ""
        targets.append(target_text)

    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    labels["input_ids"] = [
        [(label if label != tokenizer.pad_token_id else -100) for label in labels_example]
        for labels_example in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_test = test_dataset.map(prepare_test_data, batched=True)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [37]:
from torch.utils.data import DataLoader
from tqdm import tqdm # Красивая полоска загрузки
import numpy as np

def evaluate_simple(model, tokenizer_obj, test_dataset):
    # Создаем DataLoader для тестирования
    dataloader = DataLoader(test_dataset, batch_size=8)
    model.to("cuda")
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            # Переносим данные на GPU
            input_ids = batch["input_ids"].to("cuda")
            attention_mask = batch["attention_mask"].to("cuda")
            # Labels are now present in tokenized_test, used for reference in metric computation
            labels = batch["labels"].to("cuda")

            # Генерируем ответы
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=128,
                num_beams=4,
                early_stopping=True
            )

            # Декодируем
            preds = tokenizer_obj.batch_decode(outputs, skip_special_tokens=True)
            # Decode labels for metric computation, handling -100 masking
            labels_cpu = labels.cpu()
            labels_decoded = np.where(labels_cpu != -100, labels_cpu, tokenizer_obj.pad_token_id)
            labs = tokenizer_obj.batch_decode(labels_decoded, skip_special_tokens=True)

            all_preds.extend(preds)
            all_labels.extend(labs)

    # Считаем метрики вручную через squad_metric
    formatted_preds = [{"id": str(i), "prediction_text": p} for i, p in enumerate(all_preds)]
    formatted_refs = [{"id": str(i), "answers": {"text": [l], "answer_start": [0]}} for i, l in enumerate(all_labels)]

    return squad_metric.compute(predictions=formatted_preds, references=formatted_refs)

In [49]:
import torch
from tqdm import tqdm # Красивая полоска загрузки
import numpy as np

def evaluate_simple_v2(model, tokenizer_obj, dataset_list):
    model.to("cuda")
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for item in tqdm(dataset_list):
            # item["input_ids"] is now expected to be a 1D torch.Tensor
            # Add a batch dimension for the model input
            input_ids = item["input_ids"].unsqueeze(0).to("cuda")
            attention_mask = item["attention_mask"].unsqueeze(0).to("cuda")

            # Generate outputs, including num_beams and early_stopping
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=128,
                num_beams=4,
                early_stopping=True
            )

            # Decode predictions
            preds = tokenizer_obj.batch_decode(outputs, skip_special_tokens=True)

            # item["labels"] is also expected to be a 1D torch.Tensor
            # Handle -100 masking for correct decoding of reference labels
            labels_cpu = item["labels"].cpu().numpy()
            labels_decoded_for_metric = np.where(labels_cpu != -100, labels_cpu, tokenizer_obj.pad_token_id)
            labs = tokenizer_obj.decode(labels_decoded_for_metric, skip_special_tokens=True)

            all_preds.extend(preds) # preds is a list of one string
            all_labels.append(labs) # labs is a single string

    formatted_preds = [{"id": str(i), "prediction_text": p} for i, p in enumerate(all_preds)]
    formatted_refs = [{"id": str(i), "answers": {"text": [l], "answer_start": [0]}} for i, l in enumerate(all_labels)]

    return squad_metric.compute(predictions=formatted_preds, references=formatted_refs)

In [50]:
results = {}

# Убедимся, что датасет готов к работе (re-add this line)
tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 1. Необученная
results["LongT5 (необученная)"] = evaluate_simple_v2(model, tokenizer, tokenized_test)

# 2. Наша дообученная
model_finetuned = LongT5ForConditionalGeneration.from_pretrained("./t5-squad-finetuned")
model_finetuned.eval()
results["LongT5 (наша дообученная)"] = evaluate_simple_v2(model_finetuned, tokenizer, tokenized_test)

# 3. Готовая T5-base
try:
    from transformers import AutoModelForSeq2SeqLM # Ensure this import is here
    model_squad = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-base")
    tokenizer_squad = AutoTokenizer.from_pretrained("google-t5/t5-base")

    # Перетокенизируем данные для новой модели
    def tokenize_for_squad(ex):
        inputs = [f"question: {q} context: {c}" for q, c in zip(ex["question"], ex["context"])]
        model_inputs = tokenizer_squad(inputs, max_length=512, truncation=True, padding="max_length")

        target_texts = [ans["text"][0] if len(ans["text"]) > 0 else "" for ans in ex["answers"]]
        labels = tokenizer_squad(target_texts, max_length=128, truncation=True, padding="max_length")
        labels["input_ids"] = [
            [(label if label != tokenizer_squad.pad_token_id else -100) for label in labels_example]
            for labels_example in labels["input_ids"]
        ]
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    test_ds_squad = test_dataset.map(tokenize_for_squad, batched=True)
    # Set format to torch for this specific dataset as well
    test_ds_squad.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    results["T5-base (готовая)"] = evaluate_simple_v2(model_squad, tokenizer_squad, test_ds_squad)
except Exception as e:
    print(f"Ошибка: {e}")
    results["T5-base (готовая)"] = {"Exact Match": 'N/A', "F1 Score": 'N/A'}

100%|██████████| 100/100 [00:20<00:00,  4.93it/s]


Loading weights:   0%|          | 0/297 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:20<00:00,  4.80it/s]


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

100%|██████████| 100/100 [00:16<00:00,  6.09it/s]


In [52]:
import pandas as pd

df_results = pd.DataFrame(results).T
print(df_results)

                           exact_match         f1
LongT5 (необученная)              31.0  42.423038
LongT5 (наша дообученная)         31.0  42.423038
T5-base (готовая)                 70.0  82.726263


In [ ]:
'''
3 эпохи на маленьком датасете не успели поменять веса модели
объем данных был слишком мал, что даже не особо сработал судя по метрикам

Специализированное дообучение на полном объеме данных бенчмарка
дает критический прирост точности (EM 70.0 против 31.0),
что доказывает важность масштабирования обучающей выборки

качество ответов на вопросы ограничено не только архитектурой модели,
но и объемом данных, используемых для её узкоспециализированной настройки
'''